# 🗂️ Notebook 2: Typeahead / Autocomplete — Data Model & APIs


## 🛠️ Setup

```bash
cd 06-system-designs/typeahead-autocomplete
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Data we keep

### 1. Term store (popularity counts)
A mapping `term → score`. In production this is rebuilt nightly/hourly
from query logs. Here we'll pretend it's handed to us.

### 2. The serving index
A structure that, given a prefix, returns top-K terms fast.
For this notebook we use a **dict of prefixes → top-K list** — the simplest
possible precomputed index. It's the "cache every answer" brute-force version;
notebook 3 replaces it with a real trie.

### 3. Query log entry (write-only, async)
```json
{ "term": "python tutorial", "region": "us-west", "ts": 1713456789 }
```
Frontends `POST /log` for every executed search. An async pipeline aggregates
these into new popularity counts, then the trie is rebuilt.


## API surface (minimal)

Two endpoints, nothing fancy:

```http
GET  /suggest?q=pyth&k=5    → top-K suggestions for the prefix
POST /log                   → {"term": "...", "region": "..."}
```

Keep `/suggest` **stateless** and cacheable. Keep `/log` fire-and-forget.


In [ ]:
# 📦 Pydantic models — our request/response contracts.
from pydantic import BaseModel, Field

class Suggestion(BaseModel):
    term: str
    score: int

class SuggestResponse(BaseModel):
    query: str
    suggestions: list[Suggestion]

class LogEntry(BaseModel):
    term: str = Field(min_length=1, max_length=200)
    region: str | None = None

SuggestResponse(query="py", suggestions=[Suggestion(term="python", score=900)])


## A runnable FastAPI service

We build a tiny `/suggest` + `/log` service and test it *inside the notebook*
using `TestClient` — no ports, no servers, still fully exercised.

The index here is a plain dict `prefix → top-K`. Slow to build, fast to query.
In notebook 3 we swap it for a trie.


In [ ]:
# 🏗️ Build a precomputed prefix→top-K dict (simplest "index").
import unicodedata
from collections import defaultdict

def normalize(s: str) -> str:
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    return " ".join(s.lower().strip().split())

CORPUS = [
    ("python", 900), ("python tutorial", 500), ("pyramid", 100),
    ("go", 300), ("golang", 200), ("google", 800), ("google docs", 400),
    ("good morning", 250), ("good night", 150), ("news", 600),
    ("new york", 700), ("new york times", 550), ("netflix", 650),
]
CORPUS = [(normalize(t), s) for t, s in CORPUS]

def build_prefix_index(corpus, k=5):
    bucket: dict[str, list[tuple[str,int]]] = defaultdict(list)
    for term, score in corpus:
        for i in range(1, len(term) + 1):
            bucket[term[:i]].append((term, score))
    # keep only top-K per prefix
    return {p: sorted(v, key=lambda x: -x[1])[:k] for p, v in bucket.items()}

INDEX = build_prefix_index(CORPUS, k=5)
print("prefix 'goo' →", INDEX.get("goo"))
print("prefix 'new' →", INDEX.get("new"))


In [ ]:
# 🌐 FastAPI app — /suggest and /log
from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI(title="typeahead-demo")
QUERY_LOG: list[LogEntry] = []

@app.get("/suggest", response_model=SuggestResponse)
def suggest(q: str, k: int = 5):
    prefix = normalize(q)
    hits = INDEX.get(prefix, [])[:k]
    return SuggestResponse(
        query=q,
        suggestions=[Suggestion(term=t, score=s) for t, s in hits],
    )

@app.post("/log")
def log(entry: LogEntry):
    QUERY_LOG.append(entry)
    return {"ok": True, "logged": len(QUERY_LOG)}

client = TestClient(app)

print("GET /suggest?q=goo")
print(client.get("/suggest", params={"q": "goo"}).json())
print()
print("GET /suggest?q=New%20Y&k=3  (notice uppercase + partial second word)")
print(client.get("/suggest", params={"q": "New Y", "k": 3}).json())
print()
print("POST /log")
print(client.post("/log", json={"term": "kubernetes", "region": "eu"}).json())


## Why precomputed prefix→top-K is *not* what we ship

- **Memory**: every term is stored under every one of its prefixes.
  10 chars → 10 copies. Blows up fast.
- **Rebuild**: we have to rebuild the whole dict on any change.
- **No sharing**: `"google"` and `"google docs"` duplicate the `"goog"` answer.

A **trie** shares prefixes between terms and stores top-K *at each node*,
which is essentially the same idea with 10–100× less memory. That's notebook 3.


## Rate limits, caching, and sizing

- **Edge cache** short prefixes (`"a".."goo"`): a handful of prefixes absorb
  a huge share of traffic. 30-second TTL is plenty.
- **Per-IP rate limit** on `/suggest` so a bad client can't spam you.
- **Debounce on the client** (e.g., 80 ms) to skip requests for fleeting keystrokes.
- **CDN** handles `/suggest` GETs great because they're idempotent.

We'll come back to scaling and sharding in notebook 3.
